# 01. load raw data

## 0. setup

In [1]:
import gc
from pathlib import Path
 
import numpy as np
import pandas as pd

# paths
root         = Path.cwd().parent
data_raw     = root / 'data' / 'raw'
data_interim = root / 'data' / 'interim'
data_proc    = root / 'data' / 'processed'

data_interim.mkdir(parents=True, exist_ok=True)

## 1. config

In [2]:
# patent inputs
pat_raw = data_raw / 'patent'
pat_files = {
    'epo_app': pat_raw / '202602_EPO_App_reg.txt',
    'epo_ipc': pat_raw / '202602_EPO_IPC.txt',
    'tpf_epo': pat_raw / '202602_TPF_EPO.txt',
    'epo_cit': pat_raw / '202602_EPO_CIT_COUNTS.txt'
}

# output
out_pat = data_interim / 'pat_data.parquet'
print(f'inputs : {pat_raw.name}/  ->  output: {out_pat.name}')

inputs : patent/  ->  output: pat_data.parquet


## 2. patents

### 2.1 load epo applicants and ipc codes

In [4]:
# applicants
epo_app = pd.read_csv(
    pat_files['epo_app'], sep='|',
    usecols=['appln_id', 'app_nbr', 'person_id', 'ctry_code', 'app_share']
).rename(columns={'app_nbr': 'pat_nbr'})

# ipc codes
epo_ipc = pd.read_csv(pat_files['epo_ipc'], sep='|', low_memory=False).rename(columns={'IPC': 'ipc'})

print(f'epo_app: {len(epo_app):,} | epo_ipc: {len(epo_ipc):,}')

epo_app: 4,963,719 | epo_ipc: 19,247,975


In [5]:
def _dedup(df, name):
    n0 = len(df)
    out = df.drop_duplicates()
    print(f'{name}: {n0:,} -> {len(out):,} (dropped {n0 - len(out):,}, {(n0 - len(out)) / n0:.1%})')
    return out

epo_app = _dedup(epo_app, 'epo_app')
epo_ipc = _dedup(epo_ipc, 'epo_ipc')

epo_app: 4,963,719 -> 4,958,521 (dropped 5,198, 0.1%)
epo_ipc: 19,247,975 -> 19,247,975 (dropped 0, 0.0%)


### 2.2 merge applicants × ipc

In [ ]:
pat_df = epo_app.merge(epo_ipc, on='appln_id', how='inner')
pat_df['applt_id'] = pat_df['person_id'].astype('Int64').astype(str)
pat_df = pat_df.drop(columns='person_id')

del epo_app, epo_ipc; gc.collect()

print(f'merged: {len(pat_df):,} rows (patent x applicant x ipc)')

merged: 20,884,090 rows (patent x applicant x ipc)


### 2.3 drop key-missing rows

In [8]:
n0 = len(pat_df)
pat_df = pat_df.dropna(subset=['appln_id', 'applt_id', 'prio_year', 'ipc', 'ctry_code', 'app_share'])
print(f'dropna on key cols: {n0:,} -> {len(pat_df):,}')

pat_df['prio_year'] = pat_df['prio_year'].astype('Int64')

print(f"raw: {pat_df['pat_nbr'].nunique():,} patents | {len(pat_df):,} rows "
      f"| {pat_df['prio_year'].min()}-{pat_df['prio_year'].max()}")

dropna on key cols: 20,880,192 -> 20,880,192
raw: 4,592,226 patents | 20,880,192 rows | 1961-2025


### 2.5 deduplicate triadic patent families (keep earliest application)

In [ ]:
tpf_pat = (pd.read_csv(pat_files['tpf_epo'], sep='|', usecols=['Family_id', 'Appln_id'])
             .rename(columns=str.lower)
             .drop_duplicates(subset=['appln_id']))

pat_df = pat_df.merge(tpf_pat, on='appln_id', how='left')

in_tpf = pat_df['family_id'].notna()
n_appln_pre   = pat_df['appln_id'].nunique()
n_appln_tpf   = pat_df.loc[in_tpf, 'appln_id'].nunique()

keep = (pat_df.loc[in_tpf, ['family_id', 'appln_id', 'prio_year']]
        .drop_duplicates(subset=['family_id', 'appln_id'])
        .sort_values(['family_id', 'prio_year', 'appln_id'])
        .drop_duplicates('family_id', keep='first')[['family_id', 'appln_id']])
keep_set = set(keep['appln_id'])

pat_df = pat_df[(~in_tpf) | (pat_df['appln_id'].isin(keep_set))].copy()
pat_df = pat_df.drop(columns='family_id')

del keep, keep_set, tpf_pat; gc.collect()

n_appln_post = pat_df['appln_id'].nunique()
print(f'applications in a triadic family : {n_appln_tpf:,} / {n_appln_pre:,} '
      f'({n_appln_tpf / n_appln_pre:.1%})')
print(f'after tpf dedup                  : {n_appln_post:,} '
      f'(collapsed {n_appln_pre - n_appln_post:,})')
print(f'rows: {len(pat_df):,} | {pat_df["prio_year"].min()}-{pat_df["prio_year"].max()}')

applications in a triadic family : 2,203,118 / 4,592,226 (48.0%)
after tpf dedup                  : 4,275,969 (collapsed 316,257)
rows: 18,739,316 | 1961-2025


### 2.6 forward citations

In [ ]:
# forward-citation counts per EP patent
cit = pd.read_csv(pat_files['epo_cit'], sep='|',
                  usecols=['EP_Appln_id','Direct_cits_Recd','Direct_cits_Recd_in3'])

cit_key = (cit.rename(columns={'EP_Appln_id':'appln_id',
                               'Direct_cits_Recd':'cit_fwd',            # lifetime
                               'Direct_cits_Recd_in3':'cit_fwd_3yr'})   # 3-yr window
             .drop_duplicates('appln_id', keep='first'))

# patent merge
pat_df['appln_id'] = pat_df['appln_id'].astype('int64')
cit_key['appln_id'] = cit_key['appln_id'].astype('int64')
pat_df = pat_df.merge(cit_key, on='appln_id', how='left')

pat_df['cit_matched'] = pat_df['cit_fwd'].notna()

del cit, cit_key; gc.collect()

m = pat_df.drop_duplicates('appln_id')
rate = m['cit_matched'].mean()
print(f'citation match: {rate:.1%} of applications '
      f'({int(m["cit_matched"].sum()):,}/{len(m):,})')
print('cit_fwd_3yr (matched):',
      m.loc[m['cit_matched'], 'cit_fwd_3yr'].describe(percentiles=[.5, .9]).round(2).to_dict())

if rate < 0.95:
    print(f'\nWARNING: {1 - rate:.1%} of EP applications have no citation record. '
          f'Check the coverage window of {pat_files["epo_cit"].name} before using '
          f'the citation outcomes.')

citation match: 100.0% of applications (4,275,969/4,275,969)
cit_fwd_3yr (matched): {'count': 4275969.0, 'mean': 0.48, 'std': 1.24, 'min': 0.0, '50%': 0.0, '90%': 2.0, 'max': 185.0}


### 2.7 diagnostics

In [11]:
print('patents')
print(f'  rows         : {len(pat_df):,}')
print(f'  applications : {pat_df['appln_id'].nunique():,}')
print(f'  applicants   : {pat_df['applt_id'].nunique():,}')
print(f'  cit-matched  : {pat_df.drop_duplicates("appln_id")["cit_matched"].mean():.1%}')
print(f'  years        : {pat_df['prio_year'].min()}-{pat_df['prio_year'].max()}')
print(f'  countries    : {pat_df['ctry_code'].nunique()}')

patents
  rows         : 18,739,316
  applications : 4,275,969
  applicants   : 855,886
  cit-matched  : 100.0%
  years        : 1961-2025
  countries    : 208


### 2.7 save

In [ ]:
pat_df.to_parquet(out_pat, index=False)
print(f'saved: {out_pat.name} ({len(pat_df):,} rows)')

del pat_df; gc.collect()

saved: pat_data.parquet (18,739,316 rows)


0